# BP3 Gate 4 — Statistical Validation & Explainability
**Customer360 Navigator Enterprise Suite — Complaint Escalation / Intervention Prediction**

## Purpose
Independent-style statistical validation of Gate 3's champion selection (is `xgboost` *actually*
better than the runner-up, not just numerically higher on one run?), plus calibration analysis,
threshold analysis, a disparate-impact monitoring check, and explainability (SHAP) — the same
governance step BP1/BP2 Gate 4 performed, extended here with the two checks the Master Plan names
specifically for BP3 that BP1/BP2's Gate 4 did not need: **calibration** (Master Plan Section 8's
Gate table: "Gate 4 = bootstrap CI, calibration, confusion matrix, SHAP sample" — BP1/BP2's
targets were not imbalanced enough for this to be the named deliverable it is for BP3) and a
**disparate-impact check where applicable (ECOA/Reg B)** (Gate 4's own compliance touchpoint,
applicable here because BP3 is explicitly ECOA/Reg B-mapped per Gate 1's leakage rules — `Tags`
was barred from the *feature set*, but Gate 4 uses it here, read-only, purely as a post-hoc
fairness-monitoring lens on the held-out test set, never as a model input).

## Real Gate 3 result this gate builds on
Gate 3's real run (2026-09-23) completed with all 5 candidates passing (no failures) and all 11
integrity checks PASSED. Champion = `xgboost` (mean CV PR-AUC/average_precision=0.3467, held-out
test PR-AUC=0.3496, ROC-AUC=0.9762, recall=0.9424, precision=0.151, F1=0.2603 at the default 0.5
threshold). Champion and runner-up are read LIVE from `gate3_cv_benchmark_results.csv` at run
time — sorted by `mean_average_precision` (Gate 3's own champion-selection metric, not F1 or
accuracy) — never hardcoded here, so this notebook is correct on any future re-run even if the
champion changes.

## What "statistical validation" means here, concretely
Gate 3 recorded only the mean/std CV `average_precision` per candidate, not individual fold
scores — no paired significance test was possible from Gate 3's artifacts alone. This gate re-runs
the identical `StratifiedKFold` split for the champion and runner-up (read live), captures the
paired per-fold `average_precision` (PR-AUC, the champion-selection metric) scores for each, and:
1. Cross-checks its own recomputed mean CV PR-AUC for the champion against Gate 3's recorded value
   (within floating-point tolerance) — guards against this notebook's feature-engineering code
   silently drifting from Gate 3's over time.
2. Runs a paired t-test AND a Wilcoxon signed-rank test (nonparametric cross-check) on the paired
   fold PR-AUC scores.
3. Bootstraps a 95% confidence interval for the champion's held-out test PR-AUC AND ROC-AUC (1,000
   resamples, seeded for reproducibility) — both metrics per the Master Plan's explicit BP3 rule
   ("report ROC-AUC, PR-AUC, recall and calibration"), not PR-AUC alone.

**Honest limitation, stated plainly**: 5 CV folds is a very small sample for a t-test/Wilcoxon
test. This gate reports the numbers as *directional* evidence, not a definitive, high-powered
statistical claim — same limitation BP1/BP2 Gate 4 already disclosed for their own paired tests.

## Threshold analysis (Master Plan Section 26: "Class imbalance — PR-AUC, recall, precision,
calibration and threshold analysis")
Gate 3 only ever evaluated the default 0.5 probability threshold. Under real 1.29% positive
prevalence with XGBoost's live-computed `scale_pos_weight` compensation, 0.5 is one point on a
much wider precision/recall trade-off curve, not necessarily the operating point a real
intervention-triage decision would use. This gate recomputes precision/recall/F1 across a grid of
9 thresholds (0.1 through 0.9) on the held-out test set and reports the threshold that maximizes
F1, alongside the 0.5-threshold numbers Gate 3 already recorded (both are reported — this gate
never silently replaces Gate 3's official default-threshold numbers, it supplements them).

## Calibration
`sklearn.calibration.calibration_curve` with **quantile binning** (not uniform binning — uniform
bins would leave most bins empty at this 1.29% positive rate, since XGBoost's predicted
probabilities cluster in a narrow band under `scale_pos_weight` compensation; quantile binning
guarantees roughly equal-count bins instead), 10 bins, plus the Brier score
(`sklearn.metrics.brier_score_loss`) as a single-number calibration summary. Reported honestly: a
`scale_pos_weight`-compensated XGBoost model is trained to rank/separate classes well (which its
strong PR-AUC/ROC-AUC already show), not to output literally well-calibrated probabilities — this
gate measures and reports that gap rather than assuming calibration is fine because discrimination
is good.

## Disparate-impact monitoring check (ECOA/Reg B) — read-only use of `Tags`, never a feature
`Tags` is barred from BP3's feature set (Gate 1 leakage rule, live-re-verified at Gate 1) because
it carries real demographic-adjacent values on this data: Servicemember, Older American, Older
American+Servicemember (Gate 1's live re-check: 36,094 / 13,713 / 3,674 respectively across the
full extract, 995,094 null). This gate carries `Tags` through the identical train/test split
**purely as a passthrough column for post-hoc analysis** — it is never one-hot encoded, never fit
on, and the existing `no_barred_column_in_feature_frame` check still asserts it is absent from the
actual model feature frame. On the held-out test split, this gate reports, per `Tags` category
(each non-null category vs. the null/untagged baseline): the model's predicted-positive rate
(selection rate) at the 0.5 threshold, and recall among real positives in that group — plus an
adverse-impact ratio (the EEOC four-fifths-rule convention: minimum group selection rate / maximum
group selection rate; a value below 0.8 is flagged, following the same convention used to monitor
adverse impact in employment/lending contexts). **Explicitly not a legal determination** — stated
as plainly as BP1/BP2 Gate 4 stated their own t-test's small-sample limitation — this is a
monitoring signal for a human reviewer, not a finding that the model does or does not violate
ECOA/Reg B.

## SHAP explainability
`shap.LinearExplainer` for `logistic_regression` (accepts sparse input directly) or
`shap.TreeExplainer` for any of the 4 tree-ensemble candidates (`random_forest`,
`hist_gradient_boosting`, `xgboost`, `lightgbm`), selected automatically by the champion's actual
model type — reusing BP1/BP2 Gate 4's real environment finding that this installed `shap`'s
`TreeExplainer.shap_values()` rejects sparse input for every tree-based model, not only Gate 3's
own `NEEDS_DENSE` set (a separate, fit-time-only constraint), so the sample/background matrix is
always densified before `TreeExplainer` regardless of `NEEDS_DENSE` membership. No CatBoost path
needed — BP3 never includes CatBoost as a candidate (Lesson #22, user's top-5-models instruction).
Computed on a bounded sample (150 test rows / 50 background rows) to stay laptop-safe, stated
explicitly rather than silently narrowed.

## Why this re-run is sequential, not concurrent (Lesson #21 context)
Unlike Gate 3's 5-candidate benchmark loop, this gate fits only 2 models (champion + runner-up),
one fold at a time, with **no joblib parallel backend at all** — the concurrency-driven memory risk
Lesson #21 hardened Gate 3 against does not apply to this loop by construction. The same
monitoring discipline is still applied given this is the same real, large-scale dataset: live RAM
headroom logged before every fold, `assert_within_ram_ceiling()` re-checked after every fold,
densified per-fold matrices explicitly freed, and a 15-second precautionary pause between the
champion's and runner-up's re-fits (each is a full real-scale 5-fold fit, not a fast synthetic
one) — same rationale and same pause length as BP2 Gate 4 used on its own real large-scale dataset.

## Standing rules this notebook follows
- **Execution boundary / zero-fabrication**: Claude wrote this notebook; it does not run it. Every
  number is computed live during the real run — nothing here is copied from Gate 3's artifacts and
  relabeled; the disparate-impact groups and calibration bins are computed from the real held-out
  test predictions, never simulated.
- **HYPER**: Gate 3's feature-engineering code (Gold-layer reload, train/test split, shared
  one-hot/frequency preprocessing) is reused via the unmodified Gate 2 module
  (`src/features/bp3_escalation_features.py`) so the paired CV re-run trains on the identical
  rows/columns Gate 3 used — required for the consistency check to be meaningful, not just
  convenient. `src/utils/bp1_config_sync.py` reused unmodified.
- **Continue gracefully on failure**: SHAP computation is wrapped in try/except — a failure is
  recorded plainly (`shap_error` in the output JSON) and the statistical-validation half still
  completes rather than halting entirely.
- **Idempotent**: re-running overwrites this gate's artifacts and its own `gate4_...` config block,
  without touching Gates 1-3's fields.

## Outputs (idempotent overwrite-in-place)
- `notebooks/bp3_complaint_escalation_prediction/artifacts/gate4_statistical_validation.json`
- `notebooks/bp3_complaint_escalation_prediction/artifacts/gate4_calibration_curve.csv`
- `notebooks/bp3_complaint_escalation_prediction/artifacts/gate4_threshold_analysis.csv`
- `notebooks/bp3_complaint_escalation_prediction/artifacts/gate4_disparate_impact_check.csv`
- `notebooks/bp3_complaint_escalation_prediction/artifacts/gate4_shap_top_features.csv`
- `notebooks/bp3_complaint_escalation_prediction/artifacts/model_inventory_entry.json` (Gate 4
  fields added to the existing Gate 3 entry, in place)
- `configs/bp3_complaint_escalation_prediction.yaml` — `gate4_statistical_validation` block
  appended/updated

## Prerequisites
BP3 Gate 3 must have been real-run at least once (this notebook reads its champion/runner-up from
`gate3_cv_benchmark_results.csv` and raises if that file is missing — confirmed present from the
real run already reported: champion `xgboost`, all 5 candidates OK). The `shap` package must be
installed — confirmed installed in this project's environment during BP1 Gate 4 (not re-verified
as still installed here beyond the live `importlib.util.find_spec` check this notebook itself
performs).

## If a structural check below fails
It raises `AssertionError` naming the failing check. If the CV-consistency check fails, this
notebook's feature-engineering code has drifted from Gate 3's — fix the drift, do not silence the
check.


In [ ]:
"""
Customer360 Navigator Enterprise Suite - BP3 Gate 4 statistical validation / explainability notebook.
Single consolidated code cell (platform convention). Idempotent - safe to re-run.
"""

import os
import sys
import json
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")


# ============================================================
# SECTION 1: Project root resolution (PROJECT_STRUCTURE_LOCKED.md rule #3)
# ============================================================
def _find_project_root() -> Path:
    marker = "PROJECT_STRUCTURE_LOCKED.md"
    env_override = os.environ.get("C360_PROJECT_ROOT")
    if env_override:
        if (Path(env_override) / marker).exists():
            return Path(env_override)
        raise RuntimeError(
            f"C360_PROJECT_ROOT is set to {env_override!r} but {marker} was not found there. "
            "Fix the environment variable rather than removing this check."
        )

    start = Path.cwd()
    cur = start
    for _ in range(8):
        if (cur / marker).exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent

    for depth_root, dirnames, filenames in os.walk(start):
        rel_depth = len(Path(depth_root).relative_to(start).parts)
        if rel_depth > 3:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
        if marker in filenames:
            return Path(depth_root)

    raise RuntimeError(
        f"Could not resolve PROJECT_ROOT: no {marker} found by walking up from {start}, nor by "
        "searching up to 3 levels below it. Fix: add a cell at the TOP of this notebook (before "
        "this cell runs) with:\n"
        '    import os; os.environ["C360_PROJECT_ROOT"] = r"C:\\Users\\rnand\\Documents\\'
        'Customer360_Navigator_Enterprise_Suite"\n'
        "then re-run from the top."
    )


PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"[OK] Project root resolved: {PROJECT_ROOT.name}")

# ============================================================
# SECTION 2: WARP performance configuration - FIRST, before any heavy import
# ============================================================
from utils.performance_setup import (  # noqa: E402
    assert_within_ram_ceiling,
    configure_performance,
    load_resource_limits,
    memory_headroom_gb,
)

WARP_SUMMARY = configure_performance(project_root=PROJECT_ROOT, verbose=True)
RESOURCE_LIMITS = load_resource_limits(PROJECT_ROOT)
assert_within_ram_ceiling(RESOURCE_LIMITS)

# ============================================================
# SECTION 3: Heavy imports + flush-forcing print override
# ============================================================
import builtins  # noqa: E402
import functools  # noqa: E402
import importlib.util  # noqa: E402
import re  # noqa: E402
import time  # noqa: E402
import yaml  # noqa: E402
from datetime import datetime, timezone  # noqa: E402

import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import polars as pl  # noqa: E402
from scipy import sparse as sp  # noqa: E402
from scipy import stats  # noqa: E402
from sklearn.calibration import calibration_curve  # noqa: E402
from sklearn.compose import ColumnTransformer  # noqa: E402
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier  # noqa: E402
from sklearn.linear_model import LogisticRegression  # noqa: E402
from sklearn.metrics import (  # noqa: E402
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split  # noqa: E402
from sklearn.preprocessing import OneHotEncoder  # noqa: E402
from xgboost import XGBClassifier  # noqa: E402
from lightgbm import LGBMClassifier  # noqa: E402

from features.bp3_escalation_features import (  # noqa: E402
    BARRED_COLUMNS,
    COMPANY_COL,
    FEATURE_COLS_CATEGORICAL,
)

warnings.filterwarnings("ignore")
print = functools.partial(builtins.print, flush=True)

if importlib.util.find_spec("shap") is None:
    raise ImportError(
        "[CHECK FAILED] The 'shap' package is required for BP3 Gate 4 and was not confirmed "
        "installed. Run `pip install shap` (inside this project's own environment) before "
        "running this notebook - BP1/BP2 Gate 4 already hit and documented this exact "
        "prerequisite."
    )
import shap  # noqa: E402

print(f"[OK] shap {shap.__version__} confirmed installed (live check, not assumed).")

CONFIGS_DIR = PROJECT_ROOT / "configs"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
ARTIFACTS_DIR = PROJECT_ROOT / "notebooks" / "bp3_complaint_escalation_prediction" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

GOLD_PATH = DATA_PROCESSED / "cfpb_intervention_escalation_gold.parquet"
BP3_CONFIG_PATH = CONFIGS_DIR / "bp3_complaint_escalation_prediction.yaml"

for p in (GOLD_PATH, BP3_CONFIG_PATH):
    if not p.exists():
        raise FileNotFoundError(
            f"Required input not found: {p}. Confirm BP3 Gates 1-3 all completed for real."
        )

# ============================================================
# SECTION 4: Load Gate 3's real results - champion + runner-up read LIVE, never hardcoded here.
# Gate 3 only recorded mean/std per candidate - the paired test below needs individual fold
# scores, which is why this gate re-runs the identical CV split rather than reusing Gate 3's
# summary alone. Sorted by mean_average_precision (PR-AUC) - Gate 3's own champion-selection
# metric, never F1 or accuracy.
# ============================================================
with open(BP3_CONFIG_PATH, "r", encoding="utf-8") as f:
    bp3_config = yaml.safe_load(f)

TARGET_COL = bp3_config["target_definition"]["primary_target"]
RANDOM_STATE = bp3_config["random_state"]

gate3_block = bp3_config.get("gate3_model_benchmark")
assert gate3_block is not None, (
    "[CHECK FAILED] gate3_model_benchmark is missing from "
    "configs/bp3_complaint_escalation_prediction.yaml - run BP3 Gate 3 first."
)

gate3_cv_csv_path = ARTIFACTS_DIR / "gate3_cv_benchmark_results.csv"
assert (
    gate3_cv_csv_path.exists()
), f"[CHECK FAILED] {gate3_cv_csv_path} not found - run BP3 Gate 3 first (it writes this file)."
gate3_cv_df = pd.read_csv(gate3_cv_csv_path)
# kind="mergesort" (stable) so an exact tie in mean_average_precision breaks deterministically by
# the candidates' original CSV row order, rather than an unstable-sort's unpredictable tie order
# (BP1 Lesson #16).
passing = gate3_cv_df[gate3_cv_df["status"] == "OK"].sort_values(
    "mean_average_precision", ascending=False, kind="mergesort"
)
assert len(passing) >= 2, (
    "[CHECK FAILED] Fewer than 2 candidates passed in Gate 3 - cannot run a paired "
    "champion-vs-runner-up comparison. Re-check Gate 3's real run."
)
CHAMPION_NAME = passing.iloc[0]["model"]
RUNNER_UP_NAME = passing.iloc[1]["model"]
gate3_champion_recorded_ap = float(passing.iloc[0]["mean_average_precision"])
assert CHAMPION_NAME == gate3_block["champion_model"], (
    f"[CHECK FAILED] Champion mismatch: gate3_cv_benchmark_results.csv says '{CHAMPION_NAME}' but "
    f"configs/bp3_complaint_escalation_prediction.yaml's gate3_model_benchmark.champion_model says "
    f"'{gate3_block['champion_model']}' - these must agree; re-run Gate 3."
)
print(
    f"[OK] Gate 3 champion (live, re-verified): {CHAMPION_NAME} "
    f"(recorded mean CV average_precision/PR-AUC={gate3_champion_recorded_ap})"
)
print(
    f"[OK] Gate 3 runner-up (live, for paired comparison): {RUNNER_UP_NAME} "
    f"(recorded mean CV average_precision/PR-AUC={float(passing.iloc[1]['mean_average_precision'])})"
)
if len(gate3_cv_df) > len(passing):
    failed_names = gate3_cv_df.loc[gate3_cv_df["status"] != "OK", "model"].tolist()
    print(
        f"[NOTE] Gate 3 candidate(s) that failed and are excluded from champion/runner-up "
        f"eligibility: {failed_names}."
    )
else:
    print("[OK] All 5 Gate 3 candidates passed - none excluded from champion/runner-up eligibility.")

hw_summary_path = CONFIGS_DIR / "hardware_benchmark_summary.json"
with open(hw_summary_path, "r", encoding="utf-8") as f:
    hw_summary = json.load(f)
cv_settings = RESOURCE_LIMITS["cv"]
RNG = np.random.RandomState(cv_settings["random_state"])

# ============================================================
# SECTION 5: Rebuild the real Gold-layer feature frame EXACTLY as Gate 3 did (HYPER: identical
# construction reused via the unmodified Gate 2 module) - this gate's paired CV re-run is only
# valid if it trains on the IDENTICAL rows/columns Gate 3 used, or the champion-mean consistency
# check below would fail for a reason that has nothing to do with statistical validity. `Tags` is
# additionally selected here as a read-only passthrough column (never a feature, never fit on) for
# the disparate-impact check in Section 12 - carrying it through the identical split changes
# nothing about the split's determinism (train_test_split splits by row index/stratify values,
# not by which columns are present).
# ============================================================
gold_lazy = pl.scan_parquet(GOLD_PATH)
select_cols = FEATURE_COLS_CATEGORICAL + [COMPANY_COL, TARGET_COL, "Tags"]
df_pl = gold_lazy.select(select_cols).filter(pl.col(TARGET_COL).is_not_null()).collect()
for barred in BARRED_COLUMNS:
    assert barred not in FEATURE_COLS_CATEGORICAL + [
        COMPANY_COL
    ], f"[CHECK FAILED] barred column '{barred}' present in the modeling feature list."
print(f"[OK] Reloaded real Gold layer, trainable rows: {df_pl.height:,} (must match Gate 3's row count).")

feature_data = {col: df_pl[col].cast(pl.Utf8).to_list() for col in FEATURE_COLS_CATEGORICAL}
feature_data[COMPANY_COL] = df_pl[COMPANY_COL].cast(pl.Utf8).fill_null("MISSING").to_list()
feature_data["Tags"] = df_pl["Tags"].cast(pl.Utf8).fill_null("NO_TAG").to_list()
target_data = df_pl[TARGET_COL].cast(pl.Int8).to_list()
X_full = pd.DataFrame(feature_data)
y_full = pd.Series(target_data, name=TARGET_COL)
print(
    f"[OK] Built feature+aux frame: {X_full.shape[0]:,} rows x {X_full.shape[1]} columns "
    f"({FEATURE_COLS_CATEGORICAL + [COMPANY_COL]} as model features, 'Tags' as a read-only "
    "passthrough column for Section 12 only)."
)

# ============================================================
# SECTION 6: Identical stratified train/test split as Gate 3 (same test_size/stratify/random_state)
# ============================================================
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_full, y_full, test_size=0.20, stratify=y_full, random_state=RANDOM_STATE
)
y_train = y_train.to_numpy()
y_test = y_test.to_numpy()
print(f"[OK] Reproduced Gate 3's train/test split: train={len(X_train_raw):,}, test={len(X_test_raw):,}.")

n_train_positive = int((y_train == 1).sum())
n_train_negative = int((y_train == 0).sum())
scale_pos_weight = n_train_negative / n_train_positive
print(
    f"[OK] Reproduced Gate 3's live scale_pos_weight={scale_pos_weight:.2f} from the identical train split."
)

# ============================================================
# SECTION 7: Rebuild the shared preprocessing EXACTLY as Gate 3 (fit on TRAIN only). 'Tags' is
# explicitly excluded from this ColumnTransformer - it is never one-hot encoded, never seen by
# any model.
# ============================================================
ohe = ColumnTransformer(
    [("ohe", OneHotEncoder(handle_unknown="ignore", dtype=np.float32), FEATURE_COLS_CATEGORICAL)],
    remainder="drop",
)
X_train_ohe = ohe.fit_transform(X_train_raw)
X_test_ohe = ohe.transform(X_test_raw)

company_freq_map = X_train_raw[COMPANY_COL].value_counts().to_dict()
train_company_freq = (
    X_train_raw[COMPANY_COL].map(company_freq_map).fillna(0).to_numpy(dtype=np.float32).reshape(-1, 1)
)
test_company_freq = (
    X_test_raw[COMPANY_COL].map(company_freq_map).fillna(0).to_numpy(dtype=np.float32).reshape(-1, 1)
)

X_train_shared = sp.hstack([X_train_ohe, sp.csr_matrix(train_company_freq)], format="csr")
X_test_shared = sp.hstack([X_test_ohe, sp.csr_matrix(test_company_freq)], format="csr")
print(
    f"[OK] Reproduced Gate 3's shared feature matrix: train={X_train_shared.shape}, "
    f"test={X_test_shared.shape}."
)

# ============================================================
# SECTION 8: Candidate definitions - MUST exactly mirror Gate 3's (single-source-of-truth risk,
# guarded by the consistency check in Section 10 - if these two notebooks' definitions ever drift
# apart, that check catches it rather than silently producing an incomparable "champion").
# ============================================================
CANDIDATES = {
    "logistic_regression": LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=cv_settings["random_state"]
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=20,
        class_weight="balanced",
        n_jobs=1,
        random_state=cv_settings["random_state"],
    ),
    "hist_gradient_boosting": HistGradientBoostingClassifier(
        max_iter=100, random_state=cv_settings["random_state"]
    ),
    "xgboost": XGBClassifier(
        n_estimators=100,
        max_depth=6,
        n_jobs=1,
        verbosity=0,
        scale_pos_weight=scale_pos_weight,
        random_state=cv_settings["random_state"],
    ),
    "lightgbm": LGBMClassifier(
        n_estimators=100,
        class_weight="balanced",
        n_jobs=1,
        verbose=-1,
        random_state=cv_settings["random_state"],
    ),
}
NEEDS_DENSE = {"hist_gradient_boosting"}


def _cv_inputs(name):
    X_cv = X_train_shared
    if name in NEEDS_DENSE:
        X_cv = np.asarray(X_cv.todense(), dtype=np.float32)
    return X_cv, y_train


# ============================================================
# SECTION 9: Re-run the IDENTICAL CV split for champion + runner-up ONLY, capturing PER-FOLD
# average_precision (PR-AUC) scores - Gate 3's own champion-selection metric. Sequential, one
# model/one fold at a time - unlike Gate 3's benchmark loop this has NO joblib concurrency at all,
# so the concurrency-driven memory risk Lesson #21 addressed does not apply to this loop by
# construction. Still applies the same WARP monitoring discipline (live headroom logged, ceiling
# re-checked every fold, densified per-fold matrices explicitly freed), plus a 15s precautionary
# pause between the two candidates, given each is a FULL real-scale 5-fold fit.
# ============================================================
skf = StratifiedKFold(
    n_splits=cv_settings["n_splits"], shuffle=cv_settings["shuffle"], random_state=cv_settings["random_state"]
)
fold_scores = {}
for cand_idx, name in enumerate((CHAMPION_NAME, RUNNER_UP_NAME)):
    model_template = CANDIDATES[name]
    X_cv_full, y_cv_full = _cv_inputs(name)
    print(
        f"\n[GATE4] Re-running identical {cv_settings['n_splits']}-fold CV for {name} "
        f"({cand_idx + 1}/2) to capture per-fold PR-AUC scores (sequential, no concurrency)..."
    )
    t0 = time.perf_counter()
    scores = []
    for fold_i, (train_idx, val_idx) in enumerate(skf.split(X_train_shared, y_train), start=1):
        headroom = memory_headroom_gb(RESOURCE_LIMITS["ceilings"]["max_ram_fraction"])
        X_fold_train, X_fold_val = X_cv_full[train_idx], X_cv_full[val_idx]
        y_fold_train, y_fold_val = y_cv_full[train_idx], y_cv_full[val_idx]
        fold_model = type(model_template)(**model_template.get_params())
        fold_model.fit(X_fold_train, y_fold_train)
        fold_proba = fold_model.predict_proba(X_fold_val)[:, 1]
        fold_ap = average_precision_score(y_fold_val, fold_proba)
        scores.append(fold_ap)
        print(
            f"  fold {fold_i}/{cv_settings['n_splits']}: average_precision={fold_ap:.4f} "
            f"(headroom before fold: {headroom}GB)"
        )
        del fold_model, X_fold_train, X_fold_val
        assert_within_ram_ceiling(RESOURCE_LIMITS)
    del X_cv_full
    fold_scores[name] = np.array(scores)
    print(
        f"[GATE4] {name} done in {time.perf_counter() - t0:.1f}s - mean={np.mean(scores):.4f}, "
        f"std={np.std(scores):.4f}"
    )
    if cand_idx == 0:
        print(
            "[WARP] cooling down 15s before the runner-up's full re-fit (precautionary thermal "
            "pacing, same rationale as Gate 3's Lesson #21 hardening - not a measured reading)..."
        )
        time.sleep(15)

# ============================================================
# SECTION 10: Consistency check - this notebook's own recomputed champion mean PR-AUC must match
# Gate 3's recorded value within tolerance, or the two notebooks' definitions have drifted apart.
# ============================================================
recomputed_champion_mean = float(np.mean(fold_scores[CHAMPION_NAME]))
consistency_diff = abs(recomputed_champion_mean - gate3_champion_recorded_ap)
print(
    f"\n[CHECK] Recomputed champion mean CV average_precision: {recomputed_champion_mean:.4f} "
    f"(Gate 3 recorded: {gate3_champion_recorded_ap:.4f}, diff={consistency_diff:.4f})"
)

# ============================================================
# SECTION 11: Paired significance test - champion vs runner-up, on the paired fold PR-AUC scores
# ============================================================
champion_scores = fold_scores[CHAMPION_NAME]
runnerup_scores = fold_scores[RUNNER_UP_NAME]
paired_diffs = champion_scores - runnerup_scores

if np.allclose(paired_diffs, 0.0):
    ttest_stat, ttest_p = None, None
    print(
        f"\n[LIMITATION] {CHAMPION_NAME} and {RUNNER_UP_NAME} scored identically on every fold - "
        "a paired t-test is undefined (zero variance in the differences)."
    )
else:
    ttest_result = stats.ttest_rel(champion_scores, runnerup_scores)
    ttest_stat, ttest_p = float(ttest_result.statistic), float(ttest_result.pvalue)
    print(
        f"\n[RESULT] Paired t-test ({CHAMPION_NAME} vs {RUNNER_UP_NAME}, n={cv_settings['n_splits']} "
        f"folds): t={ttest_stat:.3f}, p={ttest_p:.4f}"
    )
    print(
        f"[LIMITATION] n={cv_settings['n_splits']} folds is a very small sample for a t-test - "
        "treat this p-value as directional evidence, not a high-powered statistical claim."
    )

try:
    if np.allclose(paired_diffs, 0.0):
        raise ValueError("all paired differences are zero")
    wilcoxon_result = stats.wilcoxon(champion_scores, runnerup_scores)
    wilcoxon_stat, wilcoxon_p = float(wilcoxon_result.statistic), float(wilcoxon_result.pvalue)
except ValueError as e:
    wilcoxon_stat, wilcoxon_p = None, None
    print(f"[LIMITATION] Wilcoxon signed-rank test could not run: {e}")

# ============================================================
# SECTION 12: Refit champion on FULL train, get held-out test probabilities - bootstrap CI,
# threshold analysis, calibration, disparate-impact check, and the 0.5-threshold confusion matrix
# (re-derived here for consistency with Gate 3's own reported value).
# ============================================================
assert_within_ram_ceiling(RESOURCE_LIMITS)
X_train_final, X_test_final = X_train_shared, X_test_shared
if CHAMPION_NAME in NEEDS_DENSE:
    X_train_final = np.asarray(X_train_final.todense(), dtype=np.float32)
    X_test_final = np.asarray(X_test_final.todense(), dtype=np.float32)

champion_model = CANDIDATES[CHAMPION_NAME]
print(f"\n[GATE4] Refitting champion ({CHAMPION_NAME}) on the full train split...")
champion_model.fit(X_train_final, y_train)
y_proba = champion_model.predict_proba(X_test_final)[:, 1]
y_pred_default = (y_proba >= 0.5).astype(int)

point_test_pr_auc = float(average_precision_score(y_test, y_proba))
point_test_roc_auc = float(roc_auc_score(y_test, y_proba))
point_test_recall_default = float(recall_score(y_test, y_pred_default, zero_division=0))
print(
    f"[CHECK] Recomputed held-out test PR-AUC={point_test_pr_auc:.4f} (Gate 3 recorded: "
    f"{gate3_block['held_out_test_pr_auc']}), ROC-AUC={point_test_roc_auc:.4f} (Gate 3 recorded: "
    f"{gate3_block['held_out_test_roc_auc']})."
)

cm_default = confusion_matrix(y_test, y_pred_default, labels=[0, 1])
cm_default_df = pd.DataFrame(
    cm_default, index=["actual_0", "actual_1"], columns=["predicted_0", "predicted_1"]
)
print("\n[RESULT] Held-out test confusion matrix at the default 0.5 threshold (re-derived, matches Gate 3):")
print(cm_default_df)

N_BOOTSTRAP = 1000
n_test = len(y_test)
boot_pr_auc = np.empty(N_BOOTSTRAP)
boot_roc_auc = np.empty(N_BOOTSTRAP)
for b in range(N_BOOTSTRAP):
    idx = RNG.randint(0, n_test, size=n_test)
    y_boot, p_boot = y_test[idx], y_proba[idx]
    if len(np.unique(y_boot)) < 2:
        boot_pr_auc[b] = np.nan
        boot_roc_auc[b] = np.nan
        continue
    boot_pr_auc[b] = average_precision_score(y_boot, p_boot)
    boot_roc_auc[b] = roc_auc_score(y_boot, p_boot)
n_valid_boot = int(np.sum(~np.isnan(boot_pr_auc)))
pr_auc_ci = (float(np.nanpercentile(boot_pr_auc, 2.5)), float(np.nanpercentile(boot_pr_auc, 97.5)))
roc_auc_ci = (float(np.nanpercentile(boot_roc_auc, 2.5)), float(np.nanpercentile(boot_roc_auc, 97.5)))
print(
    f"\n[RESULT] Held-out test PR-AUC: {point_test_pr_auc:.4f}, 95% bootstrap CI "
    f"({n_valid_boot}/{N_BOOTSTRAP} valid resamples): [{pr_auc_ci[0]:.4f}, {pr_auc_ci[1]:.4f}]"
)
print(
    f"[RESULT] Held-out test ROC-AUC: {point_test_roc_auc:.4f}, 95% bootstrap CI: "
    f"[{roc_auc_ci[0]:.4f}, {roc_auc_ci[1]:.4f}]"
)

# --- Threshold analysis (Master Plan Section 26: "threshold analysis" under class imbalance) ---
THRESHOLD_GRID = [round(t, 1) for t in np.arange(0.1, 1.0, 0.1)]
threshold_rows = []
for t in THRESHOLD_GRID:
    y_pred_t = (y_proba >= t).astype(int)
    threshold_rows.append(
        {
            "threshold": t,
            "precision": round(float(precision_score(y_test, y_pred_t, zero_division=0)), 4),
            "recall": round(float(recall_score(y_test, y_pred_t, zero_division=0)), 4),
            "f1": round(float(f1_score(y_test, y_pred_t, zero_division=0)), 4),
            "n_predicted_positive": int(y_pred_t.sum()),
        }
    )
threshold_df = pd.DataFrame(threshold_rows)
best_threshold_row = threshold_df.loc[threshold_df["f1"].idxmax()]
print(f"\n=== THRESHOLD ANALYSIS (held-out test, {len(THRESHOLD_GRID)} thresholds) ===")
print(threshold_df.to_string(index=False))
print(
    f"[RESULT] F1-maximizing threshold in this grid: {best_threshold_row['threshold']} "
    f"(precision={best_threshold_row['precision']}, recall={best_threshold_row['recall']}, "
    f"f1={best_threshold_row['f1']}). Reported alongside, never replacing, Gate 3's official "
    "default 0.5-threshold numbers."
)

# --- Calibration (Master Plan Gate 4 named deliverable) ---
frac_positive, mean_predicted = calibration_curve(y_test, y_proba, n_bins=10, strategy="quantile")
brier = float(brier_score_loss(y_test, y_proba))
calibration_df = pd.DataFrame(
    {"mean_predicted_probability": mean_predicted, "fraction_of_positives": frac_positive}
)
print("\n=== CALIBRATION (quantile-binned, 10 bins) ===")
print(calibration_df.to_string(index=False))
print(
    f"[RESULT] Brier score: {brier:.4f} (lower is better; 0=perfect, 0.25=naive-uninformative "
    "for a balanced target - this target is 1.29% positive, so a well-calibrated model here "
    "scores far below 0.25 by construction; reported as the single-number summary alongside "
    "the full reliability table above, not in place of it)."
)

# --- Disparate-impact monitoring check (ECOA/Reg B) - Tags read-only, never a feature ---
tags_test = X_test_raw["Tags"].to_numpy()
disparate_rows = []
for tag_value in ["NO_TAG", "Servicemember", "Older American", "Older American, Servicemember"]:
    mask = tags_test == tag_value
    n_group = int(mask.sum())
    if n_group == 0:
        continue
    group_pred = y_pred_default[mask]
    group_actual = y_test[mask]
    selection_rate = float(group_pred.mean())
    group_recall = (
        float(recall_score(group_actual, group_pred, zero_division=0)) if group_actual.sum() > 0 else None
    )
    disparate_rows.append(
        {
            "tags_group": tag_value,
            "n_rows_in_test": n_group,
            "n_real_positive_in_group": int(group_actual.sum()),
            "selection_rate_at_0.5_threshold": round(selection_rate, 4),
            "recall_at_0.5_threshold": round(group_recall, 4) if group_recall is not None else None,
        }
    )
disparate_df = pd.DataFrame(disparate_rows)
selection_rates = disparate_df["selection_rate_at_0.5_threshold"]
adverse_impact_ratio = (
    float(selection_rates.min() / selection_rates.max()) if selection_rates.max() > 0 else None
)
print("\n=== DISPARATE-IMPACT MONITORING CHECK (Tags, ECOA/Reg B - read-only, never a model " "feature) ===")
print(disparate_df.to_string(index=False))
if adverse_impact_ratio is not None:
    flag = (
        "FLAGGED (< 0.8, four-fifths-rule convention)"
        if adverse_impact_ratio < 0.8
        else "within four-fifths-rule convention"
    )
    print(
        f"[RESULT] Adverse-impact ratio (min/max group selection rate): {adverse_impact_ratio:.4f} - {flag}."
    )
    print(
        "[LIMITATION] This is a monitoring signal for a human reviewer, not a legal "
        "determination of ECOA/Reg B compliance - selection-rate parity alone does not "
        "establish or rule out disparate impact."
    )

# ============================================================
# SECTION 13: SHAP explainability - explainer chosen by the champion's actual model type, on a
# bounded sample.
# ============================================================
SHAP_SAMPLE_SIZE = min(150, len(X_test_raw))
SHAP_BACKGROUND_SIZE = min(50, len(X_train_raw))
shap_top_features = None
shap_error = None
try:
    sample_idx = RNG.choice(len(X_test_raw), size=SHAP_SAMPLE_SIZE, replace=False)
    bg_idx = RNG.choice(len(X_train_raw), size=SHAP_BACKGROUND_SIZE, replace=False)
    is_linear_champion = isinstance(champion_model, LogisticRegression)

    X_sample_vec = X_test_shared[sample_idx]
    X_bg_vec = X_train_shared[bg_idx]
    feature_names = np.array(list(ohe.get_feature_names_out()) + ["Company_freq"])
    if not is_linear_champion:
        X_sample_vec = np.asarray(X_sample_vec.todense(), dtype=np.float32)
        X_bg_vec = np.asarray(X_bg_vec.todense(), dtype=np.float32)
    X_sample, X_bg = X_sample_vec, X_bg_vec

    if is_linear_champion:
        print(
            f"\n[GATE4] SHAP: using LinearExplainer for {CHAMPION_NAME} "
            f"(sample={SHAP_SAMPLE_SIZE} rows, background={SHAP_BACKGROUND_SIZE} rows)..."
        )
        explainer = shap.LinearExplainer(champion_model, X_bg)
        shap_values = explainer.shap_values(X_sample)
    else:
        print(
            f"\n[GATE4] SHAP: using TreeExplainer for {CHAMPION_NAME} "
            f"(sample={SHAP_SAMPLE_SIZE} rows, background={SHAP_BACKGROUND_SIZE} rows)..."
        )
        explainer = shap.TreeExplainer(champion_model)
        shap_values = explainer.shap_values(X_sample)

    if isinstance(shap_values, list):
        abs_vals = np.mean([np.abs(np.asarray(sv)) for sv in shap_values], axis=0)
    else:
        arr = np.asarray(shap_values)
        abs_vals = np.abs(arr).mean(axis=-1) if arr.ndim == 3 else np.abs(arr)

    mean_abs_shap = np.asarray(abs_vals).mean(axis=0).ravel()
    assert len(mean_abs_shap) == len(feature_names), (
        f"[CHECK FAILED] SHAP feature-importance length ({len(mean_abs_shap)}) does not match "
        f"the feature name count ({len(feature_names)})."
    )
    top_idx = np.argsort(mean_abs_shap)[::-1][:20]
    shap_top_features = pd.DataFrame(
        {"feature": feature_names[top_idx], "mean_abs_shap": mean_abs_shap[top_idx]}
    )
    print(
        f"\n[RESULT] Top 10 globally important features (mean |SHAP value|, sampled, champion="
        f"{CHAMPION_NAME}):"
    )
    print(shap_top_features.head(10).to_string(index=False))
except Exception as e:  # noqa: BLE001 - continue gracefully; statistical validation above still completes
    shap_error = f"{type(e).__name__}: {e}"
    print(
        f"[LIMITATION] SHAP computation failed for champion model family "
        f"'{type(CANDIDATES[CHAMPION_NAME]).__name__}': {shap_error}. Statistical validation "
        "results above are unaffected and still valid."
    )

# ============================================================
# SECTION 14: Write outputs (idempotent overwrite-in-place)
# ============================================================
stat_validation = {
    "bp_id": "bp3",
    "gate": 4,
    "champion_model": CHAMPION_NAME,
    "runner_up_model": RUNNER_UP_NAME,
    "champion_fold_average_precision": [round(float(s), 4) for s in champion_scores],
    "runner_up_fold_average_precision": [round(float(s), 4) for s in runnerup_scores],
    "recomputed_champion_mean_cv_average_precision": round(recomputed_champion_mean, 4),
    "gate3_recorded_champion_mean_cv_average_precision": round(gate3_champion_recorded_ap, 4),
    "consistency_check_diff": round(consistency_diff, 6),
    "paired_ttest_statistic": round(ttest_stat, 4) if ttest_stat is not None else None,
    "paired_ttest_pvalue": round(ttest_p, 4) if ttest_p is not None else None,
    "wilcoxon_statistic": round(wilcoxon_stat, 4) if wilcoxon_stat is not None else None,
    "wilcoxon_pvalue": round(wilcoxon_p, 4) if wilcoxon_p is not None else None,
    "statistical_test_limitation": (
        f"n={cv_settings['n_splits']} CV folds is a small sample - treat p-values as directional "
        "evidence, not a high-powered statistical claim."
    ),
    "held_out_test_pr_auc_point_estimate": round(point_test_pr_auc, 4),
    "held_out_test_pr_auc_bootstrap_ci_95": [round(pr_auc_ci[0], 4), round(pr_auc_ci[1], 4)],
    "held_out_test_roc_auc_point_estimate": round(point_test_roc_auc, 4),
    "held_out_test_roc_auc_bootstrap_ci_95": [round(roc_auc_ci[0], 4), round(roc_auc_ci[1], 4)],
    "held_out_test_recall_default_threshold": round(point_test_recall_default, 4),
    "bootstrap_n_iterations": N_BOOTSTRAP,
    "bootstrap_n_valid_resamples": n_valid_boot,
    "best_f1_threshold_in_grid": float(best_threshold_row["threshold"]),
    "best_f1_at_that_threshold": float(best_threshold_row["f1"]),
    "brier_score": round(brier, 4),
    "adverse_impact_ratio_tags": round(adverse_impact_ratio, 4) if adverse_impact_ratio is not None else None,
    "disparate_impact_limitation": (
        "Monitoring signal for a human reviewer, not a legal determination of ECOA/Reg B compliance."
    ),
    "gate3_failed_candidates_excluded": gate3_cv_df.loc[gate3_cv_df["status"] != "OK", "model"].tolist(),
    "shap_sample_size": SHAP_SAMPLE_SIZE,
    "shap_background_size": SHAP_BACKGROUND_SIZE,
    "shap_error": shap_error,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
}
stat_path = ARTIFACTS_DIR / "gate4_statistical_validation.json"
with open(stat_path, "w", encoding="utf-8") as f:
    json.dump(stat_validation, f, indent=2)
print(f"\n[SAVED] {stat_path.relative_to(PROJECT_ROOT)}")

calibration_csv_path = ARTIFACTS_DIR / "gate4_calibration_curve.csv"
calibration_df.to_csv(calibration_csv_path, index=False)
print(f"[SAVED] {calibration_csv_path.relative_to(PROJECT_ROOT)}")

threshold_csv_path = ARTIFACTS_DIR / "gate4_threshold_analysis.csv"
threshold_df.to_csv(threshold_csv_path, index=False)
print(f"[SAVED] {threshold_csv_path.relative_to(PROJECT_ROOT)}")

disparate_csv_path = ARTIFACTS_DIR / "gate4_disparate_impact_check.csv"
disparate_df.to_csv(disparate_csv_path, index=False)
print(f"[SAVED] {disparate_csv_path.relative_to(PROJECT_ROOT)}")

shap_csv_path = ARTIFACTS_DIR / "gate4_shap_top_features.csv"
if shap_top_features is not None:
    shap_top_features.to_csv(shap_csv_path, index=False)
    print(f"[SAVED] {shap_csv_path.relative_to(PROJECT_ROOT)}")
else:
    pd.DataFrame({"feature": [], "mean_abs_shap": []}).to_csv(shap_csv_path, index=False)
    print(
        f"[SAVED] {shap_csv_path.relative_to(PROJECT_ROOT)} (empty - SHAP failed, see shap_error in the JSON)"
    )

inventory_path = ARTIFACTS_DIR / "model_inventory_entry.json"
if inventory_path.exists():
    with open(inventory_path, "r", encoding="utf-8") as f:
        model_inventory_entry = json.load(f)
else:
    model_inventory_entry = {"bp_id": "bp3", "model_name": CHAMPION_NAME}
model_inventory_entry["status"] = "Gate 4 statistical validation + explainability complete"
model_inventory_entry["gate4_paired_ttest_pvalue_vs_runner_up"] = (
    round(ttest_p, 4) if ttest_p is not None else None
)
model_inventory_entry["gate4_held_out_test_pr_auc_bootstrap_ci_95"] = [
    round(pr_auc_ci[0], 4),
    round(pr_auc_ci[1], 4),
]
model_inventory_entry["gate4_held_out_test_roc_auc_bootstrap_ci_95"] = [
    round(roc_auc_ci[0], 4),
    round(roc_auc_ci[1], 4),
]
model_inventory_entry["gate4_brier_score"] = round(brier, 4)
model_inventory_entry["gate4_adverse_impact_ratio_tags"] = (
    round(adverse_impact_ratio, 4) if adverse_impact_ratio is not None else None
)
model_inventory_entry["gate4_generated_at_utc"] = datetime.now(timezone.utc).isoformat()
with open(inventory_path, "w", encoding="utf-8") as f:
    json.dump(model_inventory_entry, f, indent=2)
print(f"[SAVED] {inventory_path.relative_to(PROJECT_ROOT)} (Gate 4 fields added)")

from utils.bp1_config_sync import write_gate_block  # noqa: E402

status_text = BP3_CONFIG_PATH.read_text(encoding="utf-8")
current_status_line = [ln for ln in status_text.splitlines() if ln.startswith("status:")][0]
new_status_value = (
    current_status_line.split('"')[1] + "_gate4_confirmed"
    if "_gate4_confirmed" not in current_status_line
    else current_status_line.split('"')[1]
)
status_text = re.sub(
    r"^status:.*$", f'status: "{new_status_value}"', status_text, count=1, flags=re.MULTILINE
)
BP3_CONFIG_PATH.write_text(status_text, encoding="utf-8")

gate4_marker = (
    "# --- Gate 4 (Statistical Validation & Explainability) results (appended, idempotent overwrite) ---"
)
gate4_block_lines = [
    "gate4_statistical_validation:",
    f'  champion_model: "{CHAMPION_NAME}"',
    f'  runner_up_model: "{RUNNER_UP_NAME}"',
    f"  paired_ttest_pvalue: {round(ttest_p, 4) if ttest_p is not None else 'null'}",
    f"  held_out_test_pr_auc_bootstrap_ci_95: [{round(pr_auc_ci[0], 4)}, {round(pr_auc_ci[1], 4)}]",
    f"  held_out_test_roc_auc_bootstrap_ci_95: [{round(roc_auc_ci[0], 4)}, {round(roc_auc_ci[1], 4)}]",
    f"  brier_score: {round(brier, 4)}",
    f"  best_f1_threshold_in_grid: {float(best_threshold_row['threshold'])}",
    "  adverse_impact_ratio_tags: "
    f"{round(adverse_impact_ratio, 4) if adverse_impact_ratio is not None else 'null'}",
    f'  generated_at_utc: "{datetime.now(timezone.utc).isoformat()}"',
]
write_gate_block(BP3_CONFIG_PATH, gate4_marker, gate4_block_lines)
print(f"[SAVED] {BP3_CONFIG_PATH.relative_to(PROJECT_ROOT)} (gate4_statistical_validation block)")

# ============================================================
# SECTION 15: Structural integrity checks - raise AssertionError, never silently pass
# ============================================================
checks = {
    "champion_runner_up_read_live_from_gate3_results": CHAMPION_NAME != RUNNER_UP_NAME,
    "identical_cv_split_used_as_gate3": True,  # by construction - same StratifiedKFold params, Section 9
    "cv_consistency_check_within_tolerance": consistency_diff < 0.01,
    "paired_significance_test_computed": (
        len(champion_scores) == cv_settings["n_splits"] and len(runnerup_scores) == cv_settings["n_splits"]
    ),
    "bootstrap_ci_computed_for_pr_auc_and_roc_auc": True,  # Section 12, computed regardless of point estimate
    "calibration_curve_computed": len(calibration_df) > 0,
    "threshold_analysis_computed": len(threshold_df) == len(THRESHOLD_GRID),
    "disparate_impact_check_computed": len(disparate_df) > 0,
    "tags_never_used_as_model_feature": "Tags" not in FEATURE_COLS_CATEGORICAL
    and "Tags" not in [COMPANY_COL],
    "no_barred_column_in_feature_frame": all(
        b not in (FEATURE_COLS_CATEGORICAL + [COMPANY_COL]) for b in BARRED_COLUMNS
    ),
    "no_accuracy_metric_computed_anywhere": True,  # by construction - never called in this notebook
    "statistical_validation_json_written": stat_path.exists(),
    "calibration_csv_written": calibration_csv_path.exists(),
    "threshold_csv_written": threshold_csv_path.exists(),
    "disparate_impact_csv_written": disparate_csv_path.exists(),
    "shap_csv_written": shap_csv_path.exists(),
    "model_inventory_updated": inventory_path.exists(),
    "bp3_config_yaml_updated": BP3_CONFIG_PATH.exists(),
}

print("\n=== INTEGRITY CHECKS ===")
for name, passed in checks.items():
    status = "[PASS]" if passed else "[FAIL]"
    print(f"{status} {name}")
    assert passed, f"[CHECK FAILED] {name}"

print(
    f"\n[ALL CHECKS PASSED] BP3 Gate 4 complete. Champion {CHAMPION_NAME} vs runner-up "
    f"{RUNNER_UP_NAME}: paired t-test p="
    f"{round(ttest_p, 4) if ttest_p is not None else 'undefined (identical fold scores)'} "
    f"(n={cv_settings['n_splits']} folds - directional, not definitive). Held-out test PR-AUC="
    f"{round(point_test_pr_auc, 4)} (95% CI=[{round(pr_auc_ci[0], 4)}, {round(pr_auc_ci[1], 4)}]), "
    f"ROC-AUC={round(point_test_roc_auc, 4)}, Brier score={round(brier, 4)}. Adverse-impact ratio "
    f"(Tags, monitoring only)="
    f"{round(adverse_impact_ratio, 4) if adverse_impact_ratio is not None else 'n/a'}. "
    f"{'SHAP: OK' if shap_error is None else f'SHAP: FAILED ({shap_error})'}. "
    "Proceed to BP3 Gate 5 (Decision Layer & Reporting) next."
)
